In [1]:
COLAB_RUNTIME = True # Set to true to set up the experiment for a Google Colab runtime

In [2]:
from pathlib import Path

def clone_repo(
    repo_url: str,
    repo_root: Path,
    git_ref: str = "main"
) -> None:
    """Clone the repository into the Google Colab VM.

    Args:
        repo_url (str): The URL of the repository to clone.
        repo_root (Path): The root directory for installing the repository.
        git_ref (str): The branch or tag to checkout.
        force_clone (bool): If True, force the clone even if the repository already exists.
    """
    %cd {str(os.path.basename(repo_root))}
    if not repo_root.exists():
        !git clone {repo_url} {str(repo_root)}
    else:
        print("Repo already exists at:", repo_root)

    %cd {str(repo_root)}
    !git fetch --all -q
    !git reset --hard origin/{git_ref}

    assert (repo_root / "utility_analysis").exists(), (
        f"Expected utility_analysis/ under {repo_root}. "
    )

if COLAB_RUNTIME:
    REPO_URL = "https://github.com/thowell332/llm-preferences.git"
    REPO_ROOT = Path( "/content/llm_preferences").expanduser()

    clone_repo(REPO_URL, REPO_ROOT)

[Errno 2] No such file or directory: '{str(os.path.basename(repo_root))}'
/content
Cloning into '/content/llm_preferences'...
remote: Enumerating objects: 570, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 570 (delta 91), reused 87 (delta 62), pack-reused 424 (from 1)
Receiving objects: 100% (570/570), 100.93 MiB | 32.01 MiB/s, done.
Resolving deltas: 100% (335/335), done.
Updating files: 100% (209/209), done.
/content/llm_preferences
HEAD is now at c746d67 Merge branch 'main' of github.com:thowell332/value-driven-behavior into main


In [3]:
%pip install --upgrade pip
%pip install -r /content/llm_preferences/requirements.txt "jedi>=0.16" "rich<14"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 55.8 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 35.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 68.9 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 101.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 121.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 94.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 107.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 118.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4

In [4]:
import os
import zipfile
from google.colab import drive
drive.mount('/content/drive')

!rsync -progress /content/drive/MyDrive/llama-32-1b-instruct-quantized.zip /content/tmp/llama-32-1b-instruct-quantized.zip
!unzip /content/tmp/llama-32-1b-instruct-quantized.zip -d /content/models/llama-32-1b-instruct-quantized

MessageError: User cancelled dfs_ephemeral authorization

In [5]:
from huggingface_hub import snapshot_download

def download_model(
    repo_id: str,
    model_dir: Path,
) -> None:
    """Download a model from Hugging Face.
    
    Args:
        model_key (str): The key of the model to download.
        repo_id (str): The ID of the repository to download the model from.
        model_dir (Path): The directory to download the model to.
    """
    model_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        local_dir=str(model_dir),
        local_dir_use_symlinks=False,
    )
    print(f"Downloaded to {model_dir}")

HF_REPO_ID = "RedHatAI/Llama-3.2-1B-Instruct-quantized.w8a8"
MODEL_KEY = "llama-32-1b-instruct"
MODEL_DIR  = Path(f"/content/models/{MODEL_KEY}").expanduser()

download_model(HF_REPO_ID, MODEL_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

recipe.yaml:   0%|          | 0.00/489 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

.gitattributes: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Downloaded to /content/models/llama-32-1b-instruct


In [12]:
MODEL_KEY = "llama-32-1b-instruct-quantized"
MODELS_PATH = REPO_ROOT / "utility_analysis" / "models.yaml"

In [14]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 338 bytes | 338.00 KiB/s, done.
From https://github.com/thowell332/llm-preferences
   c746d67..7f5da73  main       -> origin/main
Updating c746d67..7f5da73
Fast-forward
 utility_analysis/models.yaml | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [ ]:
# --- Pilot: edit these ---
import sys
from pathlib import Path

import yaml
from transformers import AutoConfig
from IPython.display import display

%matplotlib inline

if "MODEL_KEY" not in dir() or "MODELS_PATH" not in dir():
    raise RuntimeError("Run the model setup cell first (MODEL_KEY, MODELS_PATH).")

NOTEBOOK_REPO_ROOT = REPO_ROOT if COLAB_RUNTIME else MODELS_PATH.parent.parent
LP_DIR = NOTEBOOK_REPO_ROOT / "utility_analysis" / "experiments" / "linear_probes"
if str(LP_DIR) not in sys.path:
    sys.path.insert(0, str(LP_DIR))

import importlib
import notebook_runs
importlib.reload(notebook_runs)
from notebook_runs import (
    run_forced_choice_dual_probe_pilot
)

fig, info = run_forced_choice_dual_probe_pilot(
    repo_root=REPO_ROOT,
    model_key=MODEL_KEY,
    save_dir=f"results_linear_probes_forced_choice/{MODEL_KEY}",
    save_suffix="pilot_forced_choice_single_role",
    options_path="../../shared_options/options.json",
    utilities_path=f"../../shared_utilities/options_custom/{MODEL_KEY}/results_utilities_{MODEL_KEY}_helpful_assistant.json",
    roles="helpful assistant",
    layers="all",
    backend="vllm",
    probe_mode="all",
    primary_metric="mse",
)


TypeError: sequence item 14: expected str instance, PosixPath found